# 3. Calculating Relative Movements

Steps (all performed on **already filtered** signals):
1. Load the filtered signals.
2. Create new data points (mid_shoulders, mid_hips, torso_center).
3. Calculate the median shoulder width for each participant.
4. Calculate the raw relative movements (differences between points).
5. Normalize the movements by the median shoulder width.
6. Save the normalized movements as a CSV file.

## 3.1 Imports et configuration

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd().parent.parent

from config import DATA_DIR, ID_COLS, MOVEMENT_SIGNALS

## 3.2 Loading the Filtered Signals

Input: `data/movement/02_filtered_signals.csv` generated by
notebook 2.

In [3]:
in_path = DATA_DIR / "movement" / "02_filtered_signals.csv"
df_signals_filtered = pd.read_csv(in_path)

print(f"Loaded {in_path}")
print(f"Shape : {df_signals_filtered.shape}")
display(df_signals_filtered.head())

Loaded C:\Users\factory\Desktop\pons.lea\data\movement\02_filtered_signals.csv
Shape : (36238, 13)


,group,participant_id,frame,left_shoulder_x,left_wrist_x,nose_x,right_shoulder_x,right_wrist_x,left_shoulder_y,left_wrist_y,nose_y,right_shoulder_y,right_wrist_y
0,P,P1,0,0.550992,0.513236,0.494323,0.437280,0.451252,0.764520,0.506139,0.885001,0.771066,0.499538
1,P,P1,1,0.552154,0.513973,0.495482,0.437653,0.451268,0.764196,0.505323,0.885384,0.770782,0.499531
2,P,P1,2,0.553548,0.514354,0.496994,0.438644,0.451389,0.764030,0.504723,0.885796,0.771114,0.500096
3,P,P1,3,0.554926,0.514497,0.498642,0.439921,0.451550,0.763986,0.504498,0.886402,0.771870,0.501086
4,P,P1,4,0.556109,0.514495,0.500249,0.441226,0.451699,0.764020,0.504713,0.887284,0.772876,0.502355


## 3.3 Creating New Points

Based on the filtered landmarks, we calculate one derived points:
- `mid_shoulders`: the midpoint between the two shoulders
These points are stored in a new dataframe so as not to
overwrite the input signals.

In [4]:
def add_derived_points(df_signals):
    """
    Add derived points (mid_shoulders)
    from the filtered landmarks.

    Returns a new DataFrame (the input is not modified).
    """
    df = df_signals.copy()

    # Midpoint between the shoulders.
    df["mid_shoulders_x"] = (df["left_shoulder_x"] + df["right_shoulder_x"]) / 2
    df["mid_shoulders_y"] = (df["left_shoulder_y"] + df["right_shoulder_y"]) / 2

    return df


df_signals_with_derived = add_derived_points(df_signals_filtered)

derived_cols = [
    "mid_shoulders_x", "mid_shoulders_y"
]

display(df_signals_with_derived[ID_COLS + derived_cols].head())

,group,participant_id,frame,mid_shoulders_x,mid_shoulders_y
0,P,P1,0,0.494136,0.767793
1,P,P1,1,0.494904,0.767489
2,P,P1,2,0.496096,0.767572
3,P,P1,3,0.497423,0.767928
4,P,P1,4,0.498667,0.768448


## 3.4 Median Shoulder Width

For each participant, we calculate the Euclidean distance between the
two shoulders in each frame, then compute the median across all frames.
This median serves as a normalization unit specific to each
participant (accounting for distance from the camera and body type).

In [5]:
def compute_median_shoulder_width(df_signals):
    """
    Calculates the median shoulder width for each participant.
    Returns a Series indexed by participant_id.
    """
    width = np.sqrt(
        (df_signals["left_shoulder_x"] - df_signals["right_shoulder_x"]) ** 2
        + (df_signals["left_shoulder_y"] - df_signals["right_shoulder_y"]) ** 2
    )
    return (
        width.groupby(df_signals["participant_id"])
        .median()
        .rename("median_shoulder_width")
    )


median_shoulder_width = compute_median_shoulder_width(df_signals_with_derived)

print(f"Range : [{median_shoulder_width.min():.4f}, {median_shoulder_width.max():.4f}]")
display(median_shoulder_width.head(10))

Range : [0.0953, 0.1567]


participant_id
P1     0.112369
P10    0.106396
P11    0.156667
P12    0.101324
P13    0.107394
P14    0.095290
P15    0.114502
P16    0.107755
P17    0.095332
P18    0.103536
Name: median_shoulder_width, dtype: float64

## 3.5 Calculating Relative Movements (Raw, Unnormalized)

Each movement is a **difference between two filtered points**:
- `left/right_hand_vertical`: wrist − shoulder (Y),
- `left/right_hand_horizontal`: wrist − shoulder (X),
- `head_horizontal`: nose − mid_shoulders (X),
- `head_vertical`: nose − mid_shoulders (Y).

We store these signals in a **new dataframe dedicated to
movements**, separate from the signal dataframe.

In [7]:
def build_movements(df_signals):
    """
    Constructs the dataframe of relative (raw) movements.

    Input:
        df_signals: filtered signals + derived points.
    Output:
        df_movements: a new dataframe with ID_COLS + MOVEMENT_SIGNALS.
    """
    df_movements = df_signals[ID_COLS].copy()

    # Hand Movements (relative to the shoulders).
    df_movements["left_hand_vertical"] = (
        df_signals["left_wrist_y"] - df_signals["left_shoulder_y"]
    )
    df_movements["right_hand_vertical"] = (
        df_signals["right_wrist_y"] - df_signals["right_shoulder_y"]
    )
    df_movements["left_hand_horizontal"] = (
        df_signals["left_wrist_x"] - df_signals["left_shoulder_x"]
    )
    df_movements["right_hand_horizontal"] = (
        df_signals["right_wrist_x"] - df_signals["right_shoulder_x"]
    )

    # Head Movements (relative to the shoulders).
    df_movements["head_horizontal"] = (
        df_signals["nose_x"] - df_signals["mid_shoulders_x"]
    )
    df_movements["head_vertical"] = (
        df_signals["nose_y"] - df_signals["mid_shoulders_y"]
    )

    return df_movements


df_movements_raw = build_movements(df_signals_with_derived)

print(f"Shape : {df_movements_raw.shape}")
display(df_movements_raw.head())

Shape : (36238, 9)


,group,participant_id,frame,left_hand_vertical,right_hand_vertical,left_hand_horizontal,right_hand_horizontal,head_horizontal,head_vertical
0,P,P1,0,-0.258382,-0.271528,-0.037756,0.013973,0.000188,0.117207
1,P,P1,1,-0.258873,-0.271251,-0.038181,0.013615,0.000578,0.117896
2,P,P1,2,-0.259307,-0.271018,-0.039194,0.012746,0.000898,0.118224
3,P,P1,3,-0.259487,-0.270784,-0.040429,0.011629,0.001219,0.118474
4,P,P1,4,-0.259307,-0.270521,-0.041614,0.010474,0.001582,0.118836


## 3.6 Normalization of Movements

Each movement signal is divided by the median shoulder width
of the participant. The movements are then expressed in
**shoulder-width units**: independent of the distance to the
camera and body type.

In [8]:
def normalize_movements(df_movements, median_shoulder_width):
    """
    Divide each movement signal by the median shoulder width
    of the participant. Returns a new dataframe.
    """
    df = df_movements.copy()

    # Median shoulder width per row (mapped from participant_id),
    # with protection against division by zero.
    safe_width = median_shoulder_width.replace(0, np.nan)
    width_per_row = df["participant_id"].map(safe_width)

    for signal in MOVEMENT_SIGNALS:
        df[signal] = df[signal] / width_per_row

    return df


df_movements_normalized = normalize_movements(df_movements_raw, median_shoulder_width)

display(df_movements_normalized.head())

,group,participant_id,frame,left_hand_vertical,right_hand_vertical,left_hand_horizontal,right_hand_horizontal,head_horizontal,head_vertical
0,P,P1,0,-2.299396,-2.416387,-0.335999,0.124346,0.001669,1.043054
1,P,P1,1,-2.303771,-2.413923,-0.339779,0.121162,0.005145,1.049179
2,P,P1,2,-2.307631,-2.411848,-0.348793,0.113427,0.007991,1.052101
3,P,P1,3,-2.309234,-2.409766,-0.359783,0.103488,0.010848,1.054328
4,P,P1,4,-2.307629,-2.407424,-0.370330,0.093210,0.014077,1.057547


## 3.7 Verification: Complete Movement Signals

We verify how many participants have a complete time series
(without NaN) for each movement signal.

In [9]:
def count_complete_movements(df_movements, signals=MOVEMENT_SIGNALS):
    """For each movement, count the participants when the signal is complete (no missing values)."""
    rows = []
    for signal in signals:
        complete = df_movements.groupby("participant_id")[signal].apply(
            lambda s: s.notna().all()
        )
        rows.append({
            "movement": signal,
            "n_total": df_movements["participant_id"].nunique(),
            "n_complete": int(complete.sum()),
        })
    return pd.DataFrame(rows)


display(count_complete_movements(df_movements_normalized))

,movement,n_total,n_complete
0,left_hand_vertical,40,39
1,right_hand_vertical,40,39
2,left_hand_horizontal,40,39
3,right_hand_horizontal,40,39
4,head_horizontal,40,40
5,head_vertical,40,40


In [10]:
# Verify which participant is missing for left_wrist
wrist_participants = df_signals_filtered[df_signals_filtered.notna()].groupby("participant_id")[["left_wrist_x", "right_wrist_x"]].size()
print("Participants with wrist columns :")
print(wrist_participants[wrist_participants > 0].index.tolist())

# Compare with shoulder which is complete
print("\nAll participants in the filtered signals :")
print(sorted(df_signals_filtered["participant_id"].unique()))

Participants with wrist columns :
['P1', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P2', 'P20', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'SZ1', 'SZ10', 'SZ11', 'SZ12', 'SZ13', 'SZ14', 'SZ15', 'SZ16', 'SZ17', 'SZ18', 'SZ19', 'SZ2', 'SZ20', 'SZ3', 'SZ4', 'SZ5', 'SZ6', 'SZ7', 'SZ8', 'SZ9']

All participants in the filtered signals :
['P1', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P2', 'P20', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'SZ1', 'SZ10', 'SZ11', 'SZ12', 'SZ13', 'SZ14', 'SZ15', 'SZ16', 'SZ17', 'SZ18', 'SZ19', 'SZ2', 'SZ20', 'SZ3', 'SZ4', 'SZ5', 'SZ6', 'SZ7', 'SZ8', 'SZ9']


## 3.8 Visual Inspection

Tracing a standardized motion signal on a participant.

In [ ]:
#TEST_PID = df_movements_normalized["participant_id"].iloc[0]
#SIGNAL = "left_hand_vertical"

#test = df_movements_normalized[df_movements_normalized["participant_id"] == TEST_PID]

#plt.figure(figsize=(12, 4))
#plt.plot(test["frame"], test[SIGNAL])
#plt.title(f"{SIGNAL} — {TEST_PID}")
#plt.xlabel("Frame")
#plt.ylabel("Mouvement (en largeurs d'épaules)")
#plt.grid(True)
#plt.show()

## 3.9 Final Save

Output: `data/movement/03_normalized_signals.csv`.

In [11]:
out_path = DATA_DIR / "movement" / "03_normalized_signals.csv"
df_movements_normalized.to_csv(out_path, index=False)

print(f"Saved normalized movements -> {out_path}")
display(df_movements_normalized.head())

Saved normalized movements -> C:\Users\factory\Desktop\pons.lea\data\movement\03_normalized_signals.csv


,group,participant_id,frame,left_hand_vertical,right_hand_vertical,left_hand_horizontal,right_hand_horizontal,head_horizontal,head_vertical
0,P,P1,0,-2.299396,-2.416387,-0.335999,0.124346,0.001669,1.043054
1,P,P1,1,-2.303771,-2.413923,-0.339779,0.121162,0.005145,1.049179
2,P,P1,2,-2.307631,-2.411848,-0.348793,0.113427,0.007991,1.052101
3,P,P1,3,-2.309234,-2.409766,-0.359783,0.103488,0.010848,1.054328
4,P,P1,4,-2.307629,-2.407424,-0.370330,0.093210,0.014077,1.057547


In [12]:
# Verify the movements BEFORE normalization
print("Before normalization (df_movements_raw) :")
display(count_complete_movements(df_movements_raw))

print("\nAfter normalization (df_movements_normalized) :")
display(count_complete_movements(df_movements_normalized))

# See which participants have NaN values for the wrists
print("\nParticipants with left_wrist_x NaN values in df_signals_with_derived :")
left_wrist_all_nan = (
    df_signals_with_derived.groupby("participant_id")["left_wrist_x"]
    .apply(lambda s: s.isna().all())
)
print(left_wrist_all_nan[left_wrist_all_nan])

print("\nParticipants with right_wrist_x NaN values in df_signals_with_derived :")
right_wrist_all_nan = (
    df_signals_with_derived.groupby("participant_id")["right_wrist_x"]
    .apply(lambda s: s.isna().all())
)
print(right_wrist_all_nan[right_wrist_all_nan])

Before normalization (df_movements_raw) :


,movement,n_total,n_complete
0,left_hand_vertical,40,39
1,right_hand_vertical,40,39
2,left_hand_horizontal,40,39
3,right_hand_horizontal,40,39
4,head_horizontal,40,40
5,head_vertical,40,40



After normalization (df_movements_normalized) :


,movement,n_total,n_complete
0,left_hand_vertical,40,39
1,right_hand_vertical,40,39
2,left_hand_horizontal,40,39
3,right_hand_horizontal,40,39
4,head_horizontal,40,40
5,head_vertical,40,40



Participants with left_wrist_x NaN values in df_signals_with_derived :
participant_id
SZ4    True
Name: left_wrist_x, dtype: bool

Participants with right_wrist_x NaN values in df_signals_with_derived :
participant_id
SZ4    True
Name: right_wrist_x, dtype: bool
